# 03 Gold Tables (B): Products, sellers, calendar, reviews

**Owner:** Person B
**Builds:** `gold.dim_geography`, `gold.dim_products`, `gold.dim_sellers`, `gold.dim_date`, `gold.fact_reviews`, `gold.mart_seller_performance`, `gold.mart_daily_sales_weather`
**Rule:** joins happen only in gold, never in silver. Aggregate one-to-many tables to one row per key BEFORE joining.

## Build order
1. `dim_geography`
2. `dim_products`, `dim_sellers`
3. `dim_date`
4. `fact_reviews`
5. `mart_seller_performance` and `mart_daily_sales_weather` (need Person A's `fact_orders`, `fact_order_items`, `dim_customers`)

---

## 1. `gold.dim_geography`
- **Grain:** one row per `zip_code_prefix`.
- **Base:** `silver.geolocation`
- **Joins:** none (silver is already averaged to one row per prefix).
- **Columns:** `zip_code_prefix`, `lat`, `lng`, `city`, `state`.
- **Checks:** `zip_code_prefix` unique; row count equals `silver.geolocation`.

## 2. `gold.dim_products`
- **Grain:** one row per `product_id`.
- **Base:** `silver.products`
- **Joins:**

| Join | Type | Columns produced |
|---|---|---|
| `silver.category_translation` on `category_name` | `LEFT JOIN` | `category_name_english` |

- **Columns:** `product_id`, `category_name`, `category_name_english`, `name_length`, `description_length`, `photos_qty`, `weight_g`, `length_cm`, `height_cm`, `width_cm`.
- **Fallback:** if there is no translation, `category_name_english` = `category_name`; if both are NULL, use `'unknown'`.
- **Checks:** `product_id` unique; row count equals `silver.products` (a higher count means the translation table has duplicates); count of untranslated categories is reported.

## 3. `gold.dim_sellers`
- **Grain:** one row per `seller_id`.
- **Base:** `silver.sellers`
- **Joins:** none. `zip_code_prefix` stays as a key to `dim_geography`.
- **Columns:** `seller_id`, `zip_code_prefix`, `city`, `state`.
- **Checks:** `seller_id` unique; row count equals `silver.sellers`.

## 4. `gold.dim_date`
- **Grain:** one row per calendar day (2016-01-01 to 2018-12-31, covering the order dates).
- **Base:** generated calendar (recursive CTE or numbers table).
- **Calendar columns:** `date`, `year`, `quarter`, `month`, `month_name`, `week`, `day_of_week`, `is_weekend`.
- **Joins (all `LEFT JOIN` on `date`):**

| Join | Columns produced | Note |
|---|---|---|
| `silver.holidays` | `is_holiday`, `holiday_name` | `is_holiday` = 0 when there is no match |
| `silver.exchange_rates` | `usd_per_brl` | Fill gaps (weekends) with the last known rate; confirm the rule before coding |
| `silver.weather` | `sp_temp_max`, `sp_temp_min`, `sp_precipitation` | **São Paulo only.** The `sp_` prefix keeps this visible downstream |

- **Checks:** no missing days; row count equals the number of days in the range; each joined table adds no duplicate dates; weather columns are NULL outside the weather date range, not 0.
- **Power BI:** mark as the date table.

## 5. `gold.fact_reviews`
- **Grain:** one row per `order_id`.
- **Base:** `silver.reviews`
- **Joins:** none. Aggregate by `order_id` instead:
  - `AVG(review_score)` → `review_score_avg`
  - `MIN(review_score)` → `review_score_min`
  - `COUNT(*)` → `review_count`
  - `MIN(review_creation_date)` → `first_review_date`
- **Columns:** `order_id`, `review_score_avg`, `review_score_min`, `review_count`, `first_review_date`.
- **Checks:** `order_id` unique; every score is between 1 and 5; every `order_id` exists in `gold.fact_orders` (run after A finishes).

## 6. `gold.mart_seller_performance`
- **Grain:** one row per `seller_id`.
- **Base:** `gold.fact_order_items`
- **Joins:**

| Join | Type | Aggregate first | Purpose |
|---|---|---|---|
| `gold.fact_order_items` | base | one row per `seller_id` + `order_id` (`SUM(price)`, `COUNT(*)`) | seller revenue and items per order |
| `gold.fact_orders` on `order_id` | `LEFT JOIN` | none (already one row per order) | `delivery_days`, `is_late` |
| `gold.fact_reviews` on `order_id` | `LEFT JOIN` | none (already one row per order) | `review_score_avg` |
| `gold.dim_sellers` on `seller_id` | `LEFT JOIN` | none | `city`, `state` |

- **Columns:** `seller_id`, `city`, `state`, `orders`, `items_sold`, `revenue`, `avg_review_score`, `late_pct`, `avg_delivery_days`, `has_enough_orders` (1 if `orders` >= 10).
- **Logic:**
  - Aggregate items to one row per `seller_id` + `order_id` before joining to orders and reviews.
  - Count `late_pct` and `avg_delivery_days` only on delivered orders.
  - An order with several sellers is counted once for each of them, so the review and lateness are shared across those sellers. Note this in the log.
- **Why a mart:** the per-seller rollup mixes three fact tables, which is clumsy in DAX.
- **Checks:** `seller_id` unique; `SUM(revenue)` equals `SUM(price)` in `fact_order_items`; `avg_review_score` is between 1 and 5.

## 7. `gold.mart_daily_sales_weather`
- **Grain:** one row per day, **São Paulo customers only** (`state = 'SP'`).
- **Base:** `gold.dim_date`
- **Joins:**

| Join | Type | Aggregate first | Purpose |
|---|---|---|---|
| `gold.fact_orders` + `gold.dim_customers` (filter `state = 'SP'`) by `order_date` | `LEFT JOIN` | `COUNT(*)` orders per day | `orders` |
| `gold.fact_order_items` on `order_id` | `LEFT JOIN` | `SUM(price)` per `order_id`, then per day | `revenue` |
| `gold.fact_orders` + `gold.dim_customers` (SP) by `delivered_customer_date` | `LEFT JOIN` | `COUNT(*)` and `SUM(is_late)` per delivered day | `delivered_orders`, `late_orders` |
| `silver.weather` (already in `dim_date` as `sp_*`) | none | none | `sp_precipitation`, `sp_temp_max`, `sp_temp_min` |

- **Columns:** `date`, `orders`, `revenue`, `delivered_orders`, `late_orders`, `late_pct`, `sp_precipitation`, `sp_temp_max`, `sp_temp_min`, `is_rainy` (precipitation above the chosen threshold), `is_holiday`, `is_weekend`.
- **Logic:**
  - Orders and revenue are grouped by the **purchase date**; late deliveries are grouped by the **delivered date**, because weather matters when the package is on the road.
  - Aggregate each source to one row per day in its own subquery, then join them to `dim_date`, so the sources do not multiply each other.
  - Days without orders get 0, not NULL. Days without weather stay NULL.
- **Limitation:** weather covers São Paulo only. Mark every visual that uses this mart "SP only".
- **Checks:** `date` unique; `SUM(orders)` equals the number of SP orders in `fact_orders`; `SUM(revenue)` equals SP item revenue.

## Validation (run at the end)
- Row counts: gold vs silver for every table above.
- No duplicate keys.
- Orphan checks: every `fact_order_items.product_id` exists in `dim_products` and every `seller_id` exists in `dim_sellers`.

In [1]:
import pandas as pd
from sqlalchemy import text
import sys

# db.py faylının yerləşdiyi qovluğu əlavə edirik
sys.path.append("/workspaces/olist-ecommerce-analytics/notebooks")

from db import get_engine

engine = get_engine()

## 1. `gold.dim_geography`

#burada silver.geolocation cədvəlinin sətr sayları ilə poct indekslərinin unikal sayına baxırıq

In [2]:
df = pd.read_sql("""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT zip_code_prefix) AS unique_prefixes
    FROM silver.geolocation
""", engine)

df

,total_rows,unique_prefixes
0,1000163,19015


In [3]:
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.geolocation
""", engine)

df

,zip_code_prefix,lat,lng,city,state
0,4206,-23.594713,-46.603403,sao paulo,SP
1,4202,-23.578235,-46.604193,sao paulo,SP
2,4212,-23.589079,-46.601745,sao paulo,SP
3,4216,-23.598318,-46.604131,sao paulo,SP
4,4279,-23.601854,-46.604879,sao paulo,SP


#Unikal poct prefiksi üçün bir sətrdə digər sütunları qruplaşdırırıq təkrarlanan məlumatları ümümləşdiririk

In [4]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_geography"))

    # Hər poçt prefiksi üçün bir sətir yaradırıq
    conn.execute(text("""
        SELECT
            zip_code_prefix,       -- Poçt prefiksi
            AVG(lat) AS lat,        -- Orta enlik koordinatı
            AVG(lng) AS lng,        -- Orta uzunluq koordinatı
            MAX(city) AS city,      -- Şəhər adı
            MAX(state) AS state     -- Ştat kodu
        INTO gold.dim_geography    -- Yeni Gold cədvəli yaradırıq
        FROM silver.geolocation   
        GROUP BY zip_code_prefix  
    """))

In [5]:
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT zip_code_prefix) AS unique_prefixes
    FROM gold.dim_geography
""", engine)

print(df.to_string(index=False))

 total_rows  unique_prefixes
      19015            19015


## 2. `gold.dim_products`

In [6]:
# Məhsullar cədvəlindəki sütunlara baxırıq
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.products
""", engine)
df

,product_id,category_name,name_length,description_length,photos_qty,weight_g,length_cm,height_cm,width_cm
0,00066f42aeeb9f3007548bb9d3f33c38,perfumaria,53,596,6,300.0,20.0,16.0,16.0
1,00088930e925c41fd95ebfe695fd2655,automotivo,56,752,4,1225.0,55.0,10.0,26.0
2,0009406fd7479715e4bef61dd91f2462,cama_mesa_banho,50,266,2,300.0,45.0,15.0,35.0
3,001795ec6f1b187d37335e1c4704762e,consoles_games,53,274,1,600.0,30.0,20.0,20.0
4,001b237c0e9bb435f2e54071129237e9,cama_mesa_banho,42,253,1,6000.0,40.0,4.0,30.0


In [7]:
# Kateqoriya tərcümələri cədvəlindəki sütunlara baxırıq
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.category_translation
""", engine)

df

,category_name,category_name_english
0,agro_industria_e_comercio,agro_industry_and_commerce
1,alimentos,food
2,alimentos_bebidas,food_drink
3,artes,art
4,artes_e_artesanato,arts_and_craftmanship


Burada məhsul məlumatları ilə tərcümə olunmuş kateqoriya adlarını birləşdirib bir cədvələ yığırıq

In [8]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_products"))

    conn.execute(text("""
        SELECT
            p.product_id,
            p.category_name,
            COALESCE(t.category_name_english, p.category_name, 'unknown') AS category_name_english,
            p.name_length,
            p.description_length,
            p.photos_qty,
            p.weight_g,
            p.length_cm,
            p.height_cm,
            p.width_cm
        INTO gold.dim_products
        FROM silver.products p
        LEFT JOIN silver.category_translation t
            ON p.category_name = t.category_name
    """))

In [9]:
# Sətir sayını və təkrarlanan məhsul ID-lərini yoxlayırıq
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT product_id) AS unique_products
    FROM gold.dim_products
""", engine)

df

,total_rows,unique_products
0,32951,32951


## 3. `gold.dim_sellers`

In [10]:
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.sellers
""", engine)

df

,seller_id,zip_code_prefix,city,state
0,0015a82c2db000af6aaaf3ae2ecb0532,9080,santo andre,SP
1,001cca7ae9ae17fb1caed9dfb1094831,29156,cariacica,ES
2,001e6ad469a905060d959994f1b41e4f,24754,sao goncalo,RJ
3,002100f778ceb8431b7a1020ff7ab48f,14405,franca,SP
4,010da0602d7774602cd1b3f5fb7b709e,9710,sao bernardo do campo,SP


In [11]:
# Satıcı məlumatlarını Gold cədvəlinə köçürürük
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_sellers"))

    conn.execute(text("""
        SELECT
            seller_id,
            zip_code_prefix,
            city,
            state
        INTO gold.dim_sellers
        FROM silver.sellers
    """))

In [12]:
# Satıcı sayını və təkrarlanan ID-ləri yoxlayırıq
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT seller_id) AS unique_sellers
    FROM gold.dim_sellers
""", engine)

df

,total_rows,unique_sellers
0,3095,3095


## 4. `gold.dim_date`

In [13]:
# Tarixlər cədvəlini yaradırıq
with engine.begin() as conn:  # Verilənlər bazasına qoşuluruq və əməliyyatları idarə edirik
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_date"))  # Əvvəlki cədvəl varsa, silirik

    conn.execute(text("""
        WITH dates AS (  -- Tarixləri ardıcıl yaratmaq üçün müvəqqəti sorğu hazırlayırıq
            SELECT CAST('2016-01-01' AS DATE) AS [date]  -- Başlanğıc tarixini 1 yanvar 2016 təyin edirik

            UNION ALL  -- Yeni tarixləri əvvəlki nəticəyə əlavə edirik

            SELECT DATEADD(DAY, 1, [date])  -- Əvvəlki tarixə 1 gün əlavə edirik
            FROM dates  -- Əvvəl yaratdığımız tarixlərdən istifadə edirik
            WHERE [date] < '2018-12-31'  -- Son tarixə çatana qədər davam edirik
        )
        SELECT
            [date],  -- Tarixi göstəririk
            YEAR([date]) AS [year],  -- Tarixin ilini çıxarırıq
            DATEPART(QUARTER, [date]) AS [quarter],  -- İlin hansı rübü olduğunu tapırıq
            MONTH([date]) AS [month],  -- Ayın nömrəsini çıxarırıq
            DATENAME(MONTH, [date]) AS month_name,  -- Ayın adını çıxarırıq
            DATEPART(ISO_WEEK, [date]) AS [week],  -- ISO standartına görə həftənin nömrəsini tapırıq
            ((DATEPART(WEEKDAY, [date]) + @@DATEFIRST - 2) % 7) + 1 AS day_of_week,  -- Həftənin gününü bazar ertəsi=1, bazar=7 formasına gətiririk
            CASE
                WHEN ((DATEPART(WEEKDAY, [date]) + @@DATEFIRST - 2) % 7) + 1 IN (6, 7)  -- Gün şənbə və ya bazardırsa
                THEN 1 ELSE 0  -- Həftəsonudursa 1, deyilsə 0 yazırıq
            END AS is_weekend  -- Bu sütunun adını is_weekend qoyuruq
        INTO gold.dim_date  -- Nəticəni gold.dim_date adlı yeni cədvələ yazırıq
        FROM dates  -- Yaratdığımız tarix siyahısından məlumat götürürük
        OPTION (MAXRECURSION 0)  -- Təkrarlanan sorğu üçün maksimum addım məhdudiyyətini götürürük
    """))

In [14]:
pd.read_sql(
    "SELECT COUNT(*) AS total_dates FROM gold.dim_date",
    engine
)

,total_dates
0,1096


In [15]:
# Bayram günlərini göstərən sütunu cədvələ əlavə edirik
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE gold.dim_date
        ADD is_holiday BIT NULL
    """))

In [16]:
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.holidays
""", engine)

df

,holiday_date,local_name,name,is_national,state_code,types
0,2016-01-01,Confraternização Universal,New Year's Day,True,None,"[""Public""]"
1,2016-02-08,Carnaval,Carnival,True,None,"[""Bank"", ""Optional""]"
2,2016-02-09,Carnaval,Carnival,True,None,"[""Bank"", ""Optional""]"
3,2016-03-25,Sexta-feira Santa,Good Friday,True,None,"[""Public""]"
4,2016-03-27,Domingo de Páscoa,Easter Sunday,True,None,"[""Public""]"


In [17]:
# Milli bayramları tarix cədvəli ilə uyğunlaşdırırıq
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE d
        SET is_holiday = CASE
            WHEN h.holiday_date IS NOT NULL THEN 1
            ELSE 0
        END
        FROM gold.dim_date d
        LEFT JOIN silver.holidays h
            ON d.[date] = h.holiday_date
            AND h.is_national = 1
    """))

In [18]:
# Tarixlərin və milli bayramların sayını yoxlayırıq
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_dates,
        COUNT(DISTINCT [date]) AS unique_dates,
        SUM(CASE WHEN is_weekend = 1 THEN 1 ELSE 0 END) AS weekend_days,
        SUM(CASE WHEN is_holiday = 1 THEN 1 ELSE 0 END) AS holidays
    FROM gold.dim_date
""", engine)

df

,total_dates,unique_dates,weekend_days,holidays
0,1096,1096,314,39


In [19]:
# Tarix sayını və tarix aralığını yoxlayırıq
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT [date]) AS unique_dates,
        MIN([date]) AS first_date,
        MAX([date]) AS last_date
    FROM gold.dim_date
""", engine)

display(df)

,total_rows,unique_dates,first_date,last_date
0,1096,1096,2016-01-01,2018-12-31


In [20]:
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.holidays
""", engine)

display(df)

,holiday_date,local_name,name,is_national,state_code,types
0,2016-01-01,Confraternização Universal,New Year's Day,True,None,"[""Public""]"
1,2016-02-08,Carnaval,Carnival,True,None,"[""Bank"", ""Optional""]"
2,2016-02-09,Carnaval,Carnival,True,None,"[""Bank"", ""Optional""]"
3,2016-03-25,Sexta-feira Santa,Good Friday,True,None,"[""Public""]"
4,2016-03-27,Domingo de Páscoa,Easter Sunday,True,None,"[""Public""]"


In [21]:
df = pd.read_sql("""
    SELECT
        SUM(CASE WHEN is_holiday = 1 THEN 1 ELSE 0 END) AS holiday_days,
        COUNT(*) AS total_days
    FROM gold.dim_date
""", engine)

df

,holiday_days,total_days
0,39,1096


In [22]:
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.exchange_rates
""", engine)

df

,rate_date,rate_usd
0,2015-12-31,0.25250
1,2016-01-04,0.24755
2,2016-01-05,0.24977
3,2016-01-06,0.24791
4,2016-01-07,0.24699


Dollar məzənnəsini saxlamaq üçün sütun əlavə edirik

In [23]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE gold.dim_date
        ADD usd_per_brl DECIMAL(10,5) NULL
    """))

Hər tarix üçün həmin günə və ya əvvəlki ən yaxın məzənnəni götürürük

In [24]:
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE d
        SET usd_per_brl = (
            SELECT TOP 1 e.rate_usd
            FROM silver.exchange_rates e
            WHERE e.rate_date <= d.[date]
            ORDER BY e.rate_date DESC
        )
        FROM gold.dim_date d
    """))

In [25]:
df = pd.read_sql("""
    SELECT TOP 10 [date], usd_per_brl
    FROM gold.dim_date
    ORDER BY [date]
""", engine)

df

,date,usd_per_brl
0,2016-01-01,0.25250
1,2016-01-02,0.25250
2,2016-01-03,0.25250
3,2016-01-04,0.24755
4,2016-01-05,0.24977
5,2016-01-06,0.24791
6,2016-01-07,0.24699
7,2016-01-08,0.24862
8,2016-01-09,0.24862
9,2016-01-10,0.24862


In [26]:
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.weather
""", engine)

df

,weather_date,temp_max,temp_min,precipitation
0,2016-01-01,27.7,21.0,9.1
1,2016-01-02,24.5,19.8,25.7
2,2016-01-03,23.2,18.9,4.4
3,2016-01-04,25.4,17.7,0.0
4,2016-01-05,26.4,16.8,0.0


Hava məlumatlarını saxlamaq üçün 3 sütun əlavə edirik

In [27]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE gold.dim_date
        ADD sp_temp_max FLOAT NULL,
            sp_temp_min FLOAT NULL,
            sp_precipitation FLOAT NULL
    """))

Hava məlumatlarını tarixlərə uyğunlaşdırıb Gold cədvəlinə yazırıq

In [28]:
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE d
        SET
            sp_temp_max = w.temp_max,
            sp_temp_min = w.temp_min,
            sp_precipitation = w.precipitation
        FROM gold.dim_date d
        LEFT JOIN silver.weather w
            ON d.[date] = w.weather_date
    """))

In [29]:
df = pd.read_sql("""
    SELECT TOP 10
        [date],
        is_weekend,
        is_holiday,
        usd_per_brl,
        sp_temp_max,
        sp_temp_min,
        sp_precipitation
    FROM gold.dim_date
    ORDER BY [date]
""", engine)

df

,date,is_weekend,is_holiday,usd_per_brl,sp_temp_max,sp_temp_min,sp_precipitation
0,2016-01-01,0,True,0.25250,27.7,21.0,9.1
1,2016-01-02,1,False,0.25250,24.5,19.8,25.7
2,2016-01-03,1,False,0.25250,23.2,18.9,4.4
3,2016-01-04,0,False,0.24755,25.4,17.7,0.0
4,2016-01-05,0,False,0.24977,26.4,16.8,0.0
5,2016-01-06,0,False,0.24791,28.5,18.5,0.0
6,2016-01-07,0,False,0.24699,28.6,18.6,0.3
7,2016-01-08,0,False,0.24862,28.4,20.0,0.9
8,2016-01-09,1,False,0.24862,28.9,20.4,30.5
9,2016-01-10,1,False,0.24862,27.4,21.5,14.2


## 5. `gold.fact_reviews`

In [30]:
df = pd.read_sql("SELECT TOP 5 * FROM silver.reviews", engine)
df

,review_id,order_id,review_score,comment_title,comment_message,creation_date,answer_timestamp
0,00020c7512a52e92212f12d3e37513c0,e28abf2eb2f1fbcbdc2dd0cd9a561671,5,Entrega rápida!,A entrega foi super rápida e o pendente é lind...,2018-04-25,2018-04-26 14:55:36
1,00032b0141443497c898b3093690af51,04fb47576993a3cb0c12d4b25eab6e4e,5,None,None,2017-05-30,2017-06-01 23:28:55
2,000359bceee1b6fe876b30d35b5f9ef2,97868e4c884bc85f4501ae0a66181cc9,5,None,None,2018-02-06,2018-02-06 20:54:37
3,0005534973388c830bb858cfba83b17b,a589caa6892ceacc6bbf2f8cc30a8ad4,5,None,otimo produto. prazo cumprido. sabor tambem mu...,2017-09-01,2017-09-02 13:04:08
4,00055e36e9608fe969231e551983a69c,f5fea26ab547eec920e6f8ecdc5c37e4,5,None,O produto foi entregue muito antes do esperado...,2017-11-25,2017-11-27 11:37:22


Bu hissədə rəylər cədvəlində ümumi baxış olaraq neçə sətr,neçə distinct rəy id,null olan id sütunlar cəmi , min və max rəy scorlar göstərilib  

In [31]:
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT review_id) AS unique_reviews,
        SUM(CASE WHEN review_id IS NULL THEN 1 ELSE 0 END) AS null_review_id,
        SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
        SUM(CASE WHEN review_score IS NULL THEN 1 ELSE 0 END) AS null_review_score,
        MIN(review_score) AS min_score,
        MAX(review_score) AS max_score
    FROM silver.reviews
""", engine)

display(df)

,total_rows,unique_reviews,null_review_id,null_order_id,null_review_score,min_score,max_score
0,99224,98410,0,0,0,1,5


In [32]:
# Rəy məlumatlarını Gold fakt cədvəlinə köçürürük
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.fact_reviews"))

    conn.execute(text("""
        SELECT
            r.review_id,
            r.order_id,
            r.review_score,
            r.comment_title,
            r.comment_message,
            r.creation_date,
            r.answer_timestamp
        INTO gold.fact_reviews
        FROM silver.reviews r
        INNER JOIN silver.orders o
            ON r.order_id = o.order_id
    """))

In [33]:
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT CONCAT(review_id, order_id)) AS unique_combinations
    FROM gold.fact_reviews
""", engine)

display(df)

,total_rows,unique_combinations
0,99224,99224


## 6. `gold.mart_seller_performance`

In [34]:
df = pd.read_sql("SELECT TOP 5 * FROM silver.order_items", engine)
display(df)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [35]:
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id) AS unique_orders,
        COUNT(DISTINCT seller_id) AS unique_sellers,
        SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
        SUM(CASE WHEN seller_id IS NULL THEN 1 ELSE 0 END) AS null_seller_id,
        SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS null_price,
        SUM(CASE WHEN freight_value IS NULL THEN 1 ELSE 0 END) AS null_freight
    FROM silver.order_items
""", engine)

df

,total_rows,unique_orders,unique_sellers,null_order_id,null_seller_id,null_price,null_freight
0,112650,98666,3095,0,0,0,0


In [36]:
# Satıcı performansı mart cədvəlini yaradırıq
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.mart_seller_performance"))

    conn.execute(text("""
        SELECT
            s.seller_id,
            s.zip_code_prefix,
            s.city,
            s.state,
            COALESCE(x.total_orders, 0) AS total_orders,
            COALESCE(x.items_sold, 0) AS items_sold,
            COALESCE(x.total_sales, 0) AS total_sales,
            COALESCE(x.total_freight, 0) AS total_freight
        INTO gold.mart_seller_performance
        FROM silver.sellers s
        LEFT JOIN (
            SELECT
                oi.seller_id,
                COUNT(DISTINCT oi.order_id) AS total_orders,
                COUNT(*) AS items_sold,
                SUM(oi.price) AS total_sales,
                SUM(oi.freight_value) AS total_freight
            FROM silver.order_items oi
            INNER JOIN silver.orders o
                ON oi.order_id = o.order_id
            WHERE o.order_status = 'delivered'
            GROUP BY oi.seller_id
        ) x
            ON s.seller_id = x.seller_id
    """))

Yeni mart cədvəlinin sətir sayını və satıcı unikallığını yoxlayırıq

In [37]:
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT seller_id) AS unique_sellers,
        SUM(total_orders) AS total_orders,
        SUM(items_sold) AS items_sold,
        SUM(total_sales) AS total_sales
    FROM gold.mart_seller_performance
""", engine)

display(df)

,total_rows,unique_sellers,total_orders,items_sold,total_sales
0,3095,3095,97819,110197,13221498.11


## 7. `gold.mart_daily_sales_weather`

In [38]:
df = pd.read_sql("""
    SELECT TOP 5
        order_id,
        order_status,
        order_purchase_timestamp
    FROM silver.orders
    ORDER BY order_purchase_timestamp
""", engine)

display(df)

,order_id,order_status,order_purchase_timestamp
0,2e7a8482f6fb09756ca50c10d7bfc047,shipped,2016-09-04 21:15:19
1,e5fa5a7210941f7d56d0208e4e071d35,canceled,2016-09-05 00:15:34
2,809a282bbd5dbcabb6f2f724fca862ec,canceled,2016-09-13 15:24:19
3,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38
4,71303d7e93b399f5bcd537d124c0bcfa,canceled,2016-10-02 22:07:52


In [39]:
# Gündəlik satışları tarix və hava məlumatları ilə birləşdiririk
df = pd.read_sql("""
    SELECT
        d.[date] AS sales_date,
        d.[year],
        d.[month],
        d.is_weekend,
        d.is_holiday,
        d.usd_per_brl,
        d.sp_temp_max,
        d.sp_temp_min,
        d.sp_precipitation,
        COALESCE(s.total_orders, 0) AS total_orders,
        COALESCE(s.items_sold, 0) AS items_sold,
        COALESCE(s.total_sales, 0) AS total_sales,
        COALESCE(s.total_freight, 0) AS total_freight
    FROM gold.dim_date d
    LEFT JOIN (
        SELECT
            CAST(o.order_purchase_timestamp AS DATE) AS sales_date,
            COUNT(DISTINCT o.order_id) AS total_orders,
            COUNT(*) AS items_sold,
            SUM(oi.price) AS total_sales,
            SUM(oi.freight_value) AS total_freight
        FROM silver.orders o
        INNER JOIN silver.order_items oi
            ON o.order_id = oi.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY CAST(o.order_purchase_timestamp AS DATE)
    ) s
        ON d.[date] = s.sales_date
    ORDER BY d.[date]
""", engine)

df.head()

,sales_date,year,month,is_weekend,is_holiday,usd_per_brl,sp_temp_max,sp_temp_min,sp_precipitation,total_orders,items_sold,total_sales,total_freight
0,2016-01-01,2016,1,0,True,0.25250,27.7,21.0,9.1,0,0,0.0,0.0
1,2016-01-02,2016,1,1,False,0.25250,24.5,19.8,25.7,0,0,0.0,0.0
2,2016-01-03,2016,1,1,False,0.25250,23.2,18.9,4.4,0,0,0.0,0.0
3,2016-01-04,2016,1,0,False,0.24755,25.4,17.7,0.0,0,0,0.0,0.0
4,2016-01-05,2016,1,0,False,0.24977,26.4,16.8,0.0,0,0,0.0,0.0


In [40]:
# Gündəlik satış və hava məlumatlarını Gold mart cədvəlinə yazırıq
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.mart_daily_sales_weather"))

    conn.execute(text("""
        SELECT
            d.[date] AS sales_date,
            d.[year],
            d.[month],
            d.is_weekend,
            d.is_holiday,
            d.usd_per_brl,
            d.sp_temp_max,
            d.sp_temp_min,
            d.sp_precipitation,
            COALESCE(s.total_orders, 0) AS total_orders,
            COALESCE(s.items_sold, 0) AS items_sold,
            COALESCE(s.total_sales, 0) AS total_sales,
            COALESCE(s.total_freight, 0) AS total_freight
        INTO gold.mart_daily_sales_weather
        FROM gold.dim_date d
        LEFT JOIN (
            SELECT
                CAST(o.order_purchase_timestamp AS DATE) AS sales_date,
                COUNT(DISTINCT o.order_id) AS total_orders,
                COUNT(*) AS items_sold,
                SUM(oi.price) AS total_sales,
                SUM(oi.freight_value) AS total_freight
            FROM silver.orders o
            INNER JOIN silver.order_items oi
                ON o.order_id = oi.order_id
            WHERE o.order_status = 'delivered'
            GROUP BY CAST(o.order_purchase_timestamp AS DATE)
        ) s
            ON d.[date] = s.sales_date
    """))

In [41]:
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM gold.mart_daily_sales_weather
    ORDER BY sales_date
""", engine)

df

,sales_date,year,month,is_weekend,is_holiday,usd_per_brl,sp_temp_max,sp_temp_min,sp_precipitation,total_orders,items_sold,total_sales,total_freight
0,2016-01-01,2016,1,0,True,0.25250,27.7,21.0,9.1,0,0,0.0,0.0
1,2016-01-02,2016,1,1,False,0.25250,24.5,19.8,25.7,0,0,0.0,0.0
2,2016-01-03,2016,1,1,False,0.25250,23.2,18.9,4.4,0,0,0.0,0.0
3,2016-01-04,2016,1,0,False,0.24755,25.4,17.7,0.0,0,0,0.0,0.0
4,2016-01-05,2016,1,0,False,0.24977,26.4,16.8,0.0,0,0,0.0,0.0


# Part 2: Analysis (B)

All analysis cells are read-only (`q(...)` on `gold.*`). The only exception is the cell that saves a mart, labeled `# SAVE MART`. Each question gets a 2-3 sentence conclusion after the query.

## B1. Do late deliveries lower review scores?
- **Tables:** `fact_orders` (A), `fact_reviews`
- **Method:** average `review_score_avg` for late vs on-time orders; share of 1-2 star reviews by delay bucket (on time, 1-3 days late, 4-7 days, 8+ days). Optional: Mann-Whitney test in pandas/scipy.
- **Output:** DAX measures `Avg Review Score` by `is_late`; delay buckets as a calculated column.
- **Conclusion:** _write after running_

## B2. Which sellers perform best and worst?
- **Tables:** `fact_order_items`, `fact_orders`, `fact_reviews`, `dim_sellers`
- **Method:** per seller: revenue, orders, average review score, late-order share, average delivery days. Ignore sellers with very few orders (for example under 10) when ranking.
- **Output:** saved mart `gold.mart_seller_performance`.
- **Conclusion:** _write after running_

## B3. Which product categories drive revenue and satisfaction?
- **Tables:** `fact_order_items`, `dim_products`, `fact_reviews`
- **Method:** revenue, items sold, average price and average review score per `category_name_english`; top 10 by revenue; categories with the lowest scores.
- **Output:** DAX measures by `dim_products[category_name_english]`.
- **Conclusion:** _write after running_

## B4. Does rain affect orders? (São Paulo only)
- **Tables:** `fact_orders` (A), `dim_customers` (A), `dim_date`, `weather`
- **Method:** daily orders and revenue on rainy vs dry days (`precipitation > 0`, or above a chosen threshold); also late-delivery share on rainy vs dry days. Only `state = 'SP'`.
- **Output:** saved mart `gold.mart_daily_sales_weather`.
- **Limitation:** weather covers São Paulo only; mark every visual "SP only".
- **Conclusion:** _write after running_

## B5. Do holidays change sales?
- **Tables:** `fact_orders`, `dim_date`
- **Method:** average daily orders and revenue on holidays vs normal days; also the days just before and after major holidays. Compare weekdays with weekdays to avoid weekend bias.
- **Output:** DAX measures by `dim_date[is_holiday]`.
- **Conclusion:** _write after running_

- **Tables:** `fact_order_items`, `dim_date`
- **Method:** monthly revenue in BRL and in USD (`price * usd_per_brl`); does USD revenue move differently from BRL revenue?
- **Output:** DAX measure `Revenue USD`.
- **Conclusion:** _write after running_

## B1

Biz burada sifarişlərin gecikmə vaxtında verilən qiymətləndirməyə təsir edir ya yox ona baxırıq. Nəticə olaraq vaxtında çatdırılmış sifarişlərə verilən qiymətləndirmə ilə gecikmiş sifarişlərin qiymətləndirilməsinin ortalamasına baxırıq ..Qiymətləndirmənin səbəbi olmasada təsir etdiyi görülür 

In [42]:
df = pd.read_sql("""
    SELECT CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
        THEN 'Gecikib'
        ELSE 'Vaxtında'
        END AS delivery_status,
        AVG(CAST(r.review_score AS FLOAT)) AS avg_review_score
    FROM silver.orders o
    INNER JOIN gold.fact_reviews r
    ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
    AND o.order_delivered_customer_date IS NOT NULL
    AND o.order_estimated_delivery_date IS NOT NULL
    GROUP BY CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
        THEN 'Gecikib'
        ELSE 'Vaxtında'
        END """, engine)

df

,delivery_status,avg_review_score
0,Vaxtinda,4.293718
1,Gecikib,2.566494


Verilən 1 və 2 qiymətləndirmələrin neçə faizinin gecikməyə görə olduğuna baxırıq və görürükki 
vaxtında çatdırılma təxminən hər 100 rəydən 9-u narazıdır.
Gecikmiş çatdırılma təxminən hər 100 rəydən 54-ü narazıdır və buda az verilən qiymətləndirmənin böyük bir hissəsinin hardasa yarısı qədər gecikmənin payında düşür   

In [43]:
df = pd.read_sql("""
    SELECT CASE WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) <= 0
        THEN 'Vaxtinda'
        ELSE 'Gecikib'
        END AS delivery_status,
        COUNT(*) AS total_reviews,
        AVG(CASE WHEN r.review_score <= 2 THEN 1.0 ELSE 0.0 END) * 100
            AS unhappy_review_pct
    FROM silver.orders o
    INNER JOIN gold.fact_reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
    GROUP BY
        CASE WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) <= 0
        THEN 'Vaxtinda'
        ELSE 'Gecikib'
        END 
""", engine)

df

,delivery_status,total_reviews,unhappy_review_pct
0,Vaxtinda,89944,9.2791
1,Gecikib,6409,62.4122


Təxmini və faktiki çatdırılma tarixlərini müqayisə etdik. Təqvim günləri üzrə gecikmə qaydasını tətbiq edərək vaxtında çatdırılan və gecikən sifarişləri ayırdıq. Daha sonra bu qruplar üzrə müştərilərin orta rəy balını və narazı rəylərin faizini hesabladıq. Məqsədimiz çatdırılma gecikmələrinin müştəri məmnuniyyəti ilə əlaqəsini araşdırmaqdır.Bu nəticədən görə bilərik ki gecikmə vaxtı artdıqca aşağı rəylərdə artaraq davam edir.  

In [44]:
df = pd.read_sql("""
    SELECT CASE WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) <= 0
        THEN 'Vaxtında'
        WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) BETWEEN 1 AND 3
        THEN '1-3 gun gecikib'
        WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) BETWEEN 4 AND 7
        THEN '4-7 gun gecikib'
        ELSE '8+ gun gecikib'
        END AS delay_bucket,
        COUNT(*) AS total_reviews,
        AVG(CASE WHEN r.review_score <= 2 THEN 1.0 ELSE 0.0 END) * 100
        AS unhappy_review_pct
    FROM silver.orders o
    INNER JOIN gold.fact_reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
    GROUP BY CASE WHEN DATEDIFF(DAY, o.order_estimated_delivery_date, o.order_delivered_customer_date) <= 0
        THEN 'Vaxtında'
        WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) BETWEEN 1 AND 3
        THEN '1-3 gun gecikib'
        WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) BETWEEN 4 AND 7
        THEN '4-7 gun gecikib'
        ELSE '8+ gun gecikib'
        END
""", engine)

df

,delay_bucket,total_reviews,unhappy_review_pct
0,4-7 gun gecikib,1756,67.6537
1,Vaxtinda,89944,9.2791
2,8+ gun gecikib,2797,79.1919
3,1-3 gun gecikib,1856,32.1659


## B2

In [45]:
df = pd.read_sql("""
    SELECT TOP 10 *
    FROM gold.mart_seller_performance
    ORDER BY total_sales DESC
""", engine)

df

,seller_id,zip_code_prefix,city,state,total_orders,items_sold,total_sales,total_freight
0,4869f7a5dfa277a7dca6462dcf3b52b2,14840,guariba,SP,1124,1148,226987.93,20019.13
1,53243585a1d6dc2643021fd1853d8905,42738,lauro de freitas,BA,348,400,217940.44,12856.58
2,4a3ca9315b744ce9f8e9374361493884,14940,ibitinga,SP,1772,1949,196882.12,34338.31
3,fa1c13f2614d7b5c4749cbc52fecda94,13170,sumare,SP,578,579,190917.14,9916.36
4,7c67e1448b00f6e969d365cea6b010ab,8577,itaquaquecetuba,SP,973,1355,186570.05,51236.64
5,7e93a43ef30c4f03f38b393420bc753a,6429,barueri,SP,319,322,165981.49,5992.06
6,da8622b14eb17ae2831f4ac5b9dab84a,13405,piracicaba,SP,1311,1548,159816.87,24889.91
7,7a67c85e85bb2ce8582c35f2203ad736,3426,sao paulo,SP,1145,1155,139658.69,20619.83
8,1025f0e2d44d7041d6cf58b6550e0bfa,3204,sao paulo,SP,910,1420,138208.56,33716.40
9,955fee9216a65b617aa5c0531780ce60,4782,sao paulo,SP,1261,1472,131836.71,24769.77


Biz bu kodda satıcıların hər sifarişinə orta hesabla nə qədər satış məbləği düşdüyünü hesablamaq və satıcıları müqayisə etdik.Ümumi satış məbləğini ümumi sifariş sayına böldük və ən yüksək orta satış göstəricisinə malik satıcıları müəyyən etdik. Amma ilk yerlərdə yalnız bir neçə sifarişi olan satıcılar da var. Buna görə onları çoxsaylı sifariş qəbul edən satıcılarla birbaşa müqayisə etmək düzgün olmaya bilər.

In [46]:
df = pd.read_sql("""
    SELECT
        seller_id,
        total_orders,
        total_sales,
        CAST(total_sales AS DECIMAL(18, 2)) / NULLIF(total_orders, 0) AS avg_sales_per_order
    FROM gold.mart_seller_performance
    ORDER BY avg_sales_per_order DESC
""", engine)

df.head(10)

,seller_id,total_orders,total_sales,avg_sales_per_order
0,e3b4998c7a498169dc7bce44e6bb6277,1,6735.00,6735.000000
1,80ceebb4ee9b31afb6c6a916a574a1e2,1,6729.00,6729.000000
2,ee27a8f15b1dded4d213a468ba4eb391,1,6499.00,6499.000000
3,b37c4c02bda3161a7546a4e6d222d5b2,4,24075.00,6018.750000
4,585175ec331ea177fa47199e39a6170a,1,3549.00,3549.000000
5,abe021b01ba992245271b9aa422032df,2,6720.00,3360.000000
6,a00824eb9093d40e589b940ec45c4eb0,3,9399.97,3133.323333
7,e2a1ac9bf33e5549a2a4f834e70df2f8,5,14999.45,2999.890000
8,e908c0f3646e8b60375734a350d95d71,1,2951.00,2951.000000
9,d63c73efd41eb002280e7ec831424edb,2,5598.00,2799.000000


Ən azı 100 sifarişi olan satıcılar siyahısına baxırıqki bu az sifarişli satıcıların yüksək orta göstəricilərinin müqayisəyə təsirini azaldacaq. Onlar arasında birinci satıcının sifariş başına orta satış məbləği ən yüksək idi 626,27 və 348 sifarişlədir amma bu satıcının daha çox mənfəət qazandığını sübut etmir. 

In [47]:
df = pd.read_sql("""
    SELECT
        seller_id,
        total_orders,
        total_sales,
        CAST(total_sales AS DECIMAL(18, 2)) / NULLIF(total_orders, 0) AS avg_sales_per_order
    FROM gold.mart_seller_performance
    WHERE total_orders >= 100
    ORDER BY avg_sales_per_order DESC
""", engine)

df.head(10)

,seller_id,total_orders,total_sales,avg_sales_per_order
0,53243585a1d6dc2643021fd1853d8905,348,217940.44,626.265632
1,7e93a43ef30c4f03f38b393420bc753a,319,165981.49,520.318150
2,edb1ef5e36e0c8cd84eb3c9b003e486d,165,79144.90,479.666061
3,f5a590cf36251cf1162ea35bef76fe84,113,51232.78,453.387434
4,ccc4bbb5f32a6ab2b7066a4130f114e3,184,72926.72,396.340870
5,25c5c91f63607446a97b143d2d535d31,156,54208.78,347.492179
6,5dceca129747e92ff8ef7a997dc4f8ca,322,111126.73,345.114068
7,fa1c13f2614d7b5c4749cbc52fecda94,578,190917.14,330.306471
8,522620dcb18a6b31cd7bdf73665113a9,172,56690.49,329.595872
9,17a053fcb14bd219540cbde0df490be0,124,40353.24,325.429355


Ən çox sifarişi olan ilk 10 satıcını müəyyənləşdiririk

In [48]:
df = pd.read_sql("""
    SELECT
        seller_id,
        total_orders,
        total_sales,
        CAST(total_sales AS DECIMAL(18, 2)) / NULLIF(total_orders, 0) AS avg_sales_per_order
    FROM gold.mart_seller_performance
    ORDER BY total_orders DESC
""", engine)

df.head(10)

,seller_id,total_orders,total_sales,avg_sales_per_order
0,6560211a19b47992c3666cc44a7e94c0,1819,120702.83,66.356696
1,4a3ca9315b744ce9f8e9374361493884,1772,196882.12,111.107291
2,cc419e0650a3c5ba77189a1882b7556a,1651,101090.96,61.230139
3,1f50f920176fa81dab994f9023523100,1399,106655.71,76.237105
4,da8622b14eb17ae2831f4ac5b9dab84a,1311,159816.87,121.904554
5,955fee9216a65b617aa5c0531780ce60,1261,131836.71,104.549334
6,7a67c85e85bb2ce8582c35f2203ad736,1145,139658.69,121.972655
7,ea8482cd71df3c1969d7b9473ff13abc,1132,36696.76,32.417633
8,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226987.93,201.946557
9,3d871de0142ce09b7081e2b9d1733cb1,1064,93407.40,87.788910


Ən çox satışı olan ilk 10 satıcını müəyyənləşdiririk

In [49]:
df = pd.read_sql("""
    SELECT
        seller_id,
        total_orders,
        total_sales,
        CAST(total_sales AS DECIMAL(18, 2)) / NULLIF(total_orders, 0) AS avg_sales_per_order
    FROM gold.mart_seller_performance
    ORDER BY total_sales DESC
""", engine)

df.head(10)

,seller_id,total_orders,total_sales,avg_sales_per_order
0,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226987.93,201.946557
1,53243585a1d6dc2643021fd1853d8905,348,217940.44,626.265632
2,4a3ca9315b744ce9f8e9374361493884,1772,196882.12,111.107291
3,fa1c13f2614d7b5c4749cbc52fecda94,578,190917.14,330.306471
4,7c67e1448b00f6e969d365cea6b010ab,973,186570.05,191.747225
5,7e93a43ef30c4f03f38b393420bc753a,319,165981.49,520.318150
6,da8622b14eb17ae2831f4ac5b9dab84a,1311,159816.87,121.904554
7,7a67c85e85bb2ce8582c35f2203ad736,1145,139658.69,121.972655
8,1025f0e2d44d7041d6cf58b6550e0bfa,910,138208.56,151.877538
9,955fee9216a65b617aa5c0531780ce60,1261,131836.71,104.549334


Burada axtaririq ən yüksək ümumi satış edən satıcıların daşınma məbləği də yüksəkdirmi?  satış və daşınma göstəricilərinin necə dəyişdiyini müqayisə etmək üçün baxırıq.
3-cü satıcının ümumi daşınma məbləği ilk üçlükdə ən yüksəkdir 34.338,31

In [50]:
df = pd.read_sql("""
    SELECT
        seller_id,
        total_orders,
        total_sales,
        total_freight,
        CAST(total_sales AS DECIMAL(18, 2)) / NULLIF(total_orders, 0) AS avg_sales_per_order
    FROM gold.mart_seller_performance
    ORDER BY total_sales DESC
""", engine)

df.head(10)

,seller_id,total_orders,total_sales,total_freight,avg_sales_per_order
0,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226987.93,20019.13,201.946557
1,53243585a1d6dc2643021fd1853d8905,348,217940.44,12856.58,626.265632
2,4a3ca9315b744ce9f8e9374361493884,1772,196882.12,34338.31,111.107291
3,fa1c13f2614d7b5c4749cbc52fecda94,578,190917.14,9916.36,330.306471
4,7c67e1448b00f6e969d365cea6b010ab,973,186570.05,51236.64,191.747225
5,7e93a43ef30c4f03f38b393420bc753a,319,165981.49,5992.06,520.318150
6,da8622b14eb17ae2831f4ac5b9dab84a,1311,159816.87,24889.91,121.904554
7,7a67c85e85bb2ce8582c35f2203ad736,1145,139658.69,20619.83,121.972655
8,1025f0e2d44d7041d6cf58b6550e0bfa,910,138208.56,33716.40,151.877538
9,955fee9216a65b617aa5c0531780ce60,1261,131836.71,24769.77,104.549334


Cədvəldə satıcıları ümumi satış məbləğinə görə sıralayıb sonra hər satıcı üçün bir sifarişə düşən orta daşınma məbləğini hesablayırıq.

1-ci satıcı Ən yüksək ümumi satışa malikdir və orta daşınma məbləği 17.81.
2-ci satıcı Satış məbləği birinci satıcıya yaxındır, amma orta daşınma məbləği 36.94.
5-ci satıcı İlk 10 satıcı arasında orta daşınma məbləği ən yüksək olan satıcıdır 52.66.

In [51]:
df = pd.read_sql("""
    SELECT
        seller_id,
        total_orders,
        total_sales,
        total_freight,
        CAST(total_freight AS DECIMAL(18, 2)) / NULLIF(total_orders, 0) AS avg_freight_per_order
    FROM gold.mart_seller_performance
    ORDER BY total_sales DESC
""", engine)

df.head(10)

,seller_id,total_orders,total_sales,total_freight,avg_freight_per_order
0,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226987.93,20019.13,17.810614
1,53243585a1d6dc2643021fd1853d8905,348,217940.44,12856.58,36.944195
2,4a3ca9315b744ce9f8e9374361493884,1772,196882.12,34338.31,19.378279
3,fa1c13f2614d7b5c4749cbc52fecda94,578,190917.14,9916.36,17.156332
4,7c67e1448b00f6e969d365cea6b010ab,973,186570.05,51236.64,52.658417
5,7e93a43ef30c4f03f38b393420bc753a,319,165981.49,5992.06,18.783887
6,da8622b14eb17ae2831f4ac5b9dab84a,1311,159816.87,24889.91,18.985439
7,7a67c85e85bb2ce8582c35f2203ad736,1145,139658.69,20619.83,18.008585
8,1025f0e2d44d7041d6cf58b6550e0bfa,910,138208.56,33716.40,37.050989
9,955fee9216a65b617aa5c0531780ce60,1261,131836.71,24769.77,19.642958


burada hər sifarişə ortalama nə qədər daşınma haqqı çıxır ona baxarıq

In [52]:
df = pd.read_sql("""
    SELECT
        seller_id,
        total_orders,
        total_sales,
        total_freight,
        CAST(total_freight AS DECIMAL(18, 2)) / NULLIF(total_orders, 0) AS avg_freight_per_order
    FROM gold.mart_seller_performance
    ORDER BY total_orders DESC
""", engine)

df.head(10)

,seller_id,total_orders,total_sales,total_freight,avg_freight_per_order
0,6560211a19b47992c3666cc44a7e94c0,1819,120702.83,27348.11,15.034695
1,4a3ca9315b744ce9f8e9374361493884,1772,196882.12,34338.31,19.378279
2,cc419e0650a3c5ba77189a1882b7556a,1651,101090.96,24846.03,15.049079
3,1f50f920176fa81dab994f9023523100,1399,106655.71,35057.23,25.058778
4,da8622b14eb17ae2831f4ac5b9dab84a,1311,159816.87,24889.91,18.985439
5,955fee9216a65b617aa5c0531780ce60,1261,131836.71,24769.77,19.642958
6,7a67c85e85bb2ce8582c35f2203ad736,1145,139658.69,20619.83,18.008585
7,ea8482cd71df3c1969d7b9473ff13abc,1132,36696.76,17291.87,15.275504
8,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226987.93,20019.13,17.810614
9,3d871de0142ce09b7081e2b9d1733cb1,1064,93407.40,22108.25,20.778430


Bu analiz satıcılar üzrə çatdırılma xərclərinin satış məbləğinə nisbətini müqayisə etməyə imkan verir. Göstəricinin yüksək olması çatdırılma xərcinin satışa nisbətən daha böyük pay tutduğunu göstərir. Aşağı olması isə çatdırılma xərcinin satışa nisbətən daha kiçik pay tutduğunu göstərir.

In [53]:
df = pd.read_sql("""
    SELECT
        seller_id,
        total_orders,
        total_sales,
        total_freight,
        CAST(total_freight AS DECIMAL(18, 2)) * 100
            / NULLIF(total_sales, 0) AS freight_to_sales_pct
    FROM gold.mart_seller_performance
    ORDER BY total_sales DESC
""", engine)

df.head(10)

,seller_id,total_orders,total_sales,total_freight,freight_to_sales_pct
0,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226987.93,20019.13,8.819469
1,53243585a1d6dc2643021fd1853d8905,348,217940.44,12856.58,5.899125
2,4a3ca9315b744ce9f8e9374361493884,1772,196882.12,34338.31,17.441051
3,fa1c13f2614d7b5c4749cbc52fecda94,578,190917.14,9916.36,5.194065
4,7c67e1448b00f6e969d365cea6b010ab,973,186570.05,51236.64,27.462414
5,7e93a43ef30c4f03f38b393420bc753a,319,165981.49,5992.06,3.610077
6,da8622b14eb17ae2831f4ac5b9dab84a,1311,159816.87,24889.91,15.574019
7,7a67c85e85bb2ce8582c35f2203ad736,1145,139658.69,20619.83,14.764445
8,1025f0e2d44d7041d6cf58b6550e0bfa,910,138208.56,33716.40,24.395305
9,955fee9216a65b617aa5c0531780ce60,1261,131836.71,24769.77,18.788219


Satıcıların performansının təhlili göstərdi ki, ümumi satış məbləği, sifariş sayı və çatdırılma xərclərinin satışdakı payı satıcılar arasında fərqlənir. Bəzi satıcılar yüksək satış gəliri əldə etdiyi halda, çatdırılma xərclərinin satışdakı payı nisbətən aşağıdır. Digər satıcılarda isə bu pay daha yüksəkdir və çatdırılma xərclərinin əlavə araşdırılmasına ehtiyac ola bilər. Bu təhlil satıcıların satış göstəricilərini müqayisə etməyə və çatdırılma xərclərinə diqqət tələb edən halları müəyyənləşdirməyə kömək edir.

## B3

In [54]:
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.order_items
""", engine)

df

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


Bu analizdə hər kateqoriya üzrə ümumi gəliri satılan məhsulların sayını və orta məhsul qiymətini hesablayırıq.Nəticə olaraq health_beauty kateqoriyası 1,26 milyon satış gəliri ilə ilk yeri tutub. bed_bath_table 11 115 məhsulla ən çox satılan kateqoriya, watches_gifts isə 201,14 orta qiymətlə ən yüksək göstəriciyə malik kateqoriya olub. Bu nəticələr satış gəlirinin həm məhsul sayından, həm də orta qiymətdən asılı olduğunu göstərir.

In [55]:
df = pd.read_sql("""
    SELECT
        p.category_name_english,
        SUM(oi.price) AS total_revenue,
        COUNT(*) AS items_sold,
        AVG(oi.price) AS avg_price
    FROM silver.order_items oi
    INNER JOIN gold.dim_products p
        ON oi.product_id = p.product_id
    GROUP BY p.category_name_english
    ORDER BY total_revenue DESC
""", engine)

df.head(10)

,category_name_english,total_revenue,items_sold,avg_price
0,health_beauty,1258681.34,9670,130.163530
1,watches_gifts,1205005.68,5991,201.135983
2,bed_bath_table,1036988.68,11115,93.296327
3,sports_leisure,988048.97,8641,114.344285
4,computers_accessories,911954.32,7827,116.513903
5,furniture_decor,729762.49,8334,87.564493
6,cool_stuff,635290.85,3796,167.357968
7,housewares,632248.66,6964,90.788147
8,auto,592720.11,4235,139.957523
9,garden_tools,485256.46,4347,111.630195


Hansı kateqoriyaların müştərilərdən daha yüksək hansılarının isə daha aşağı rəy balı aldığını müəyyən etmək istəyirik.Bu kod hər sifarişdəki rəyləri məhsul kateqoriyaları ilə əlaqələndirir və kateqoriya üzrə orta rəy balını hesablayır.Müştəri rəylərinin təhlili nəticəsində cds_dvds_musicals kateqoriyası 4,64 orta balla ən yüksək, security_and_services isə 2,50 balla ən aşağı göstəriciyə malikdir.Yalnız burada bir məqama diqqət yetirmək lazımdır.məsələn 4,64 bal  cəmi 14 rəy qeydə alınıb, 2,50 bal lakin cəmi 2 rəy var.Bu nəticələr bizi yanılda bilər

In [56]:
df = pd.read_sql("""
    SELECT
        p.category_name_english,
        COUNT(r.review_score) AS total_reviews,
        AVG(CAST(r.review_score AS FLOAT)) AS avg_review_score
    FROM silver.order_items oi
    INNER JOIN gold.dim_products p
        ON oi.product_id = p.product_id
    INNER JOIN gold.fact_reviews r
        ON oi.order_id = r.order_id
    GROUP BY p.category_name_english
    ORDER BY avg_review_score DESC
""", engine)

df

,category_name_english,total_reviews,avg_review_score
0,cds_dvds_musicals,14,4.642857
1,fashion_childrens_clothes,8,4.500000
2,books_general_interest,549,4.446266
3,costruction_tools_tools,99,4.444444
4,flowers,31,4.419355
...,...,...,...
69,office_furniture,1687,3.493183
70,pc_gamer,9,3.333333
71,portateis_cozinha_e_preparadores_de_alimentos,15,3.266667
72,diapers_and_hygiene,39,3.256410


Əsaslı nəticə almaq üçün ən az 50 rəy üzərində olan qiymətləndirməyə baxırıq və distinct review_id əlavə edək ki, rəy sayını unikal hesablayaq.
Kateqoriyalar üzrə müştəri rəylərinin təhlili göstərdi ki books_general_interest kateqoriyası 4,46 orta rəy balı ilə ən yüksək nəticəyə malikdir. office_furniture kateqoriyası isə 3,62 balla ən aşağı göstəricini göstərib.

In [57]:
df = pd.read_sql("""
    WITH order_categories AS (
        SELECT DISTINCT
            oi.order_id,
            p.category_name_english
        FROM silver.order_items oi
        INNER JOIN gold.dim_products p
            ON oi.product_id = p.product_id
    )
    SELECT
        oc.category_name_english,
        COUNT(DISTINCT r.review_id) AS total_reviews,
        AVG(CAST(r.review_score AS FLOAT)) AS avg_review_score
    FROM order_categories oc
    INNER JOIN gold.fact_reviews r
        ON oc.order_id = r.order_id
    GROUP BY oc.category_name_english
    HAVING COUNT(DISTINCT r.review_id) > 50
    ORDER BY avg_review_score DESC
""", engine)

df


,category_name_english,total_reviews,avg_review_score
0,books_general_interest,508,4.462598
1,costruction_tools_tools,94,4.425532
2,books_technical,259,4.405405
3,food_drink,226,4.385965
4,luggage_accessories,1030,4.331068
5,books_imported,53,4.320755
6,small_appliances_home_oven_and_coffee,75,4.293333
7,food,445,4.276404
8,stationery,2294,4.244136
9,cine_photo,66,4.242424


bu analizde hansı kateqoriyaların daha çox gəlir gətirdiyini və həmin kateqoriyalarla əlaqəli sifarişlərdə müştərilərin orta rəy balının necə olduğunu müqayisə edirik.

In [58]:
df = pd.read_sql("""
    WITH category_sales AS 
    (SELECT p.category_name_english,
    SUM(oi.price) AS total_revenue,
    COUNT(*) AS items_sold
    FROM silver.order_items oi
    INNER JOIN gold.dim_products p
    ON oi.product_id = p.product_id
    GROUP BY p.category_name_english
    ),
    order_categories AS (
        SELECT DISTINCT oi.order_id,
        p.category_name_english
        FROM silver.order_items oi
        INNER JOIN gold.dim_products p
        ON oi.product_id = p.product_id),
    category_reviews AS (
        SELECT oc.category_name_english,COUNT(DISTINCT r.review_id) AS total_reviews,
        AVG(CAST(r.review_score AS FLOAT)) AS avg_review_score
        FROM order_categories oc
        INNER JOIN gold.fact_reviews r
        ON oc.order_id = r.order_id
        GROUP BY oc.category_name_english
        HAVING COUNT(DISTINCT r.review_id) > 50)
    SELECT s.category_name_english, s.total_revenue, s.items_sold, r.total_reviews,
        ROUND(r.avg_review_score, 2) AS avg_review_score
    FROM category_sales s
    INNER JOIN category_reviews r
        ON s.category_name_english = r.category_name_english
    ORDER BY s.total_revenue DESC
""", engine)

df.head(10)

,category_name_english,total_revenue,items_sold,total_reviews,avg_review_score
0,health_beauty,1258681.34,9670,8767,4.18
1,watches_gifts,1205005.68,5991,5582,4.07
2,bed_bath_table,1036988.68,11115,9324,3.97
3,sports_leisure,988048.97,8641,7656,4.17
4,computers_accessories,911954.32,7827,6641,4.03
5,furniture_decor,729762.49,8334,6404,4.01
6,cool_stuff,635290.85,3796,3608,4.17
7,housewares,632248.66,6964,5854,4.14
8,auto,592720.11,4235,3879,4.09
9,garden_tools,485256.46,4347,3505,4.13


Satış və müştəri rəylərinin müqayisəsi göstərdi ki, health_beauty kateqoriyası ən yüksək gəlir əldə edib. bed_bath_table kateqoriyası ən çox satılan məhsul sayına malik olsa da, orta rəy balı digər bəzi yüksək gəlirli kateqoriyalardan aşağıdır. sports_leisure kateqoriyası isə yüksək satış gəlirini yaxşı müştəri rəyi ilə birləşdirir. Bu nəticələr satış gəliri ilə müştəri məmnuniyyətinin kateqoriyalar arasında fərqləndiyini göstərir.

## B4 
B4-ün məqsədi
São Paulo (SP) ştatında yağışlı və yağışsız günlərdə:
Sifariş sayı dəyişirmi?
Yağışlı günlərdə gecikmiş çatdırılmaların payı artırmı? 
suallarına cavab axtaracıq

In [59]:
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM gold.mart_daily_sales_weather
""", engine)

df

,sales_date,year,month,is_weekend,is_holiday,usd_per_brl,sp_temp_max,sp_temp_min,sp_precipitation,total_orders,items_sold,total_sales,total_freight
0,2017-03-19,2017,3,1,False,0.32203,20.0,15.9,2.2,56,59,6649.34,1078.74
1,2018-04-15,2018,4,1,False,0.29341,19.7,17.2,1.2,219,260,30171.31,5973.93
2,2017-05-18,2017,5,0,False,0.31899,19.9,14.9,14.9,124,139,18205.14,2596.28
3,2017-02-24,2017,2,0,False,0.32374,29.5,19.4,2.6,60,70,6630.12,1149.08
4,2017-06-10,2017,6,1,False,0.30639,17.7,7.8,0.0,72,79,7941.56,1525.44


Biz baxırıq ilk olaraq Sao Paulo-da yağışlı və yağışsız günlərdə orta sifariş sayı fərqlənirmi

Sao Paulo üzrə aparılan ilkin təhlildə yağışlı günlərdə orta sifariş sayı 91,92, yağışsız günlərdə isə 81,76 olub. Orta gündəlik satış da yağışlı günlərdə daha yüksək qeydə alınıb. Bu nəticələr yağışlı günlərlə sifariş sayı və satış gəliri arasında əlaqə olduğunu göstərir, lakin səbəb-nəticə əlaqəsini təsdiqləmir.

In [60]:
df = pd.read_sql("""
    SELECT
        CASE
            WHEN sp_precipitation > 0 THEN 'Rainy'
            ELSE 'Dry'
        END AS weather_type,
        COUNT(*) AS total_days,
        AVG(CAST(total_orders AS FLOAT)) AS avg_orders,
        AVG(CAST(total_sales AS FLOAT)) AS avg_sales
    FROM gold.mart_daily_sales_weather
    GROUP BY
        CASE
            WHEN sp_precipitation > 0 THEN 'Rainy'
            ELSE 'Dry'
        END
""", engine)

df

,weather_type,total_days,avg_orders,avg_sales
0,Dry,420,81.764286,11318.500548
1,Rainy,676,91.918639,12526.224675


Biz burada sifarişin tarixinə uyğun hava məlumatını tapırıq. Faktiki çatdırılma tarixi təxmini tarixdən gecdirsə, sifarişi gecikmiş sayırıq.Yağışlı və yağışsız günlər üçün sifariş sayını, gecikmiş sifariş sayını və gecikmə faizini hesablayırıq.

Sao Paulo üzrə aparılan təhlildə yağışsız günlərdə sifarişlərin 5,30%-i, yağışlı günlərdə isə 9,67%-i gecikib. Beləliklə, yağışlı günlərdə gecikmə faizi 4,37 faiz bəndi daha yüksək olub. Bu nəticə yağışlı günlərlə çatdırılma gecikmələri arasında əlaqə olduğunu göstərir.

In [61]:
df = pd.read_sql("""
    SELECT
        CASE
            WHEN w.sp_precipitation > 0 THEN 'Rainy'
            ELSE 'Dry'
        END AS weather_type,
        COUNT(*) AS total_orders,
        SUM(CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 1 ELSE 0
        END) AS late_orders,
        AVG(CAST(CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 1.0 ELSE 0.0
        END AS FLOAT)) * 100 AS late_order_pct
    FROM silver.orders o
    INNER JOIN gold.mart_daily_sales_weather w
        ON CAST(o.order_purchase_timestamp AS DATE) = w.sales_date
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
    GROUP BY
        CASE
            WHEN w.sp_precipitation > 0 THEN 'Rainy'
            ELSE 'Dry'
        END
""", engine)

df

,weather_type,total_orders,late_orders,late_order_pct
0,Dry,34335,1819,5.297801
1,Rainy,62135,6007,9.667659


## B5
Burada biz bayram günlərində satışlar necə dəyişir? sual;na cavab axtaraciq

kodda bayram gününü müəyyən edir,
hər qrupa düşən günlərin sayını hesablayır,
orta sifariş sayını tapır,
orta satış gəlirini hesablayırıq

Bayram və adi günlərin müqayisəsi göstərdi ki, bayram günlərində orta sifariş sayı təxminən 28%, orta gündəlik satış gəliri isə 30% aşağı olub. Bu nəticə seçilmiş məlumat dövründə bayram günlərində satış aktivliyinin daha aşağı olduğunu göstərir.

In [62]:
df = pd.read_sql("""
    SELECT
        CASE
            WHEN is_holiday = 1 THEN 'Holiday'
            ELSE 'Normal day'
        END AS day_type,
        COUNT(*) AS total_days,
        AVG(CAST(total_orders AS FLOAT)) AS avg_orders,
        AVG(CAST(total_sales AS FLOAT)) AS avg_sales
    FROM gold.mart_daily_sales_weather
    GROUP BY
        CASE
            WHEN is_holiday = 1 THEN 'Holiday'
            ELSE 'Normal day'
        END
""", engine)

df

,day_type,total_days,avg_orders,avg_sales
0,Normal day,1057,88.924314,12197.505686
1,Holiday,39,63.717949,8429.092308


Bu kodda bayramdan əvvəlki və sonrakı günlərdə satışların dəyişib-dəyişmədiyini yoxlayırıq
Bayram günlərinin təhlili göstərdi ki bayram günü və bayramdan əvvəlki günlərdə orta satış gəliri digər günlərlə müqayisədə aşağı olub. Bayramdan sonrakı günlərdə isə sifariş sayı və satış gəliri artıb. Bu nəticələr bayram dövründə satış aktivliyinin dəyişdiyini göstərir. 

In [64]:
df = pd.read_sql("""
    WITH holiday_dates AS (
        SELECT sales_date
        FROM gold.mart_daily_sales_weather
        WHERE is_holiday = 1
    ),
    daily_data AS (
        SELECT
            d.sales_date,
            d.total_orders,
            d.total_sales,
            CASE
                WHEN d.is_holiday = 1 THEN 'Holiday'
                WHEN EXISTS (
                    SELECT 1
                    FROM holiday_dates h
                    WHERE h.sales_date = DATEADD(DAY, 1, d.sales_date)
                ) THEN 'Day before holiday'
                WHEN EXISTS (
                    SELECT 1
                    FROM holiday_dates h
                    WHERE h.sales_date = DATEADD(DAY, -1, d.sales_date)
                ) THEN 'Day after holiday'
                ELSE 'Other days'
            END AS day_type
        FROM gold.mart_daily_sales_weather d
    )
    SELECT
        day_type,
        COUNT(*) AS total_days,
        AVG(CAST(total_orders AS FLOAT)) AS avg_orders,
        AVG(CAST(total_sales AS FLOAT)) AS avg_sales
    FROM daily_data
    GROUP BY day_type
    ORDER BY day_type
""", engine)

df

,day_type,total_days,avg_orders,avg_sales
0,Day after holiday,33,80.696970,11005.925152
1,Day before holiday,35,63.628571,9244.142857
2,Holiday,39,63.717949,8429.092308
3,Other days,989,90.094034,12341.782588


Son olaraq bayram və hava şəraitinə görə sifariş ve satışların paylanmasına baxırıq.

In [65]:
df = pd.read_sql("""
    SELECT
        CASE
            WHEN is_holiday = 1 AND sp_precipitation > 0
                THEN 'Rainy holiday'
            WHEN is_holiday = 1 AND sp_precipitation = 0
                THEN 'Dry holiday'
            WHEN is_holiday = 0 AND sp_precipitation > 0
                THEN 'Rainy normal day'
            ELSE 'Dry normal day'
        END AS day_type,
        COUNT(*) AS total_days,
        AVG(CAST(total_orders AS FLOAT)) AS avg_orders,
        AVG(CAST(total_sales AS FLOAT)) AS avg_sales
    FROM gold.mart_daily_sales_weather
    GROUP BY
        CASE
            WHEN is_holiday = 1 AND sp_precipitation > 0
                THEN 'Rainy holiday'
            WHEN is_holiday = 1 AND sp_precipitation = 0
                THEN 'Dry holiday'
            WHEN is_holiday = 0 AND sp_precipitation > 0
                THEN 'Rainy normal day'
            ELSE 'Dry normal day'
        END
    ORDER BY day_type
""", engine)

df

,day_type,total_days,avg_orders,avg_sales
0,Dry holiday,13,75.692308,10721.336923
1,Dry normal day,407,81.958231,11337.574570
2,Rainy holiday,26,57.730769,7282.970000
3,Rainy normal day,650,93.286154,12735.954862


Bayram və hava şəraitinin birgə təhlili göstərdiki yağışlı adi günlərdə orta sifariş sayı yağışsız adi günlərdən yüksək olub. Bayram günlərində isə yağışlı hava daha aşağı sifariş sayı və satış gəliri ilə müşahidə edilib. Xüsusilə yağışlı bayram günlərində orta sifariş sayı 57,73, orta satış gəliri isə 7.282,97 olub. Bu nəticələr hava şəraiti və bayram günlərinin satış göstəriciləri ilə əlaqəli ola biləcəyini göstərir.